# SEMIR LiTS Two-Stage Pipeline

**Goal**: Replace GT CSV dependency in VKG pipeline with SEMIR graph pooling.

**Approach**: Crop to liver ROI, protect suspicious intensity regions, coarsen with Rust crate, evaluate oracle diagnostics.

**4 modes to compare**:
- A: Full CT, no protection
- B: Liver crop only  
- C: Liver crop + intensity-based protected region
- D: Liver crop + GT tumor protection (upper bound)

In [1]:
import numpy as np
import os, re, time, json
import fastloops
from scipy.ndimage import binary_dilation

DATA_ROOT = "/scratch/ud3d4/acm_data/Data"
RESULTS_DIR = "/home/ud3d4/Desktop/SWOG/results/semir_lits_twostage"
os.makedirs(RESULTS_DIR, exist_ok=True)

HU_MIN, HU_MAX = -50, 250
np.random.seed(42)

def pr(msg=""):
    print(msg, flush=True)

## Step 1: Data loading + discovery

In [2]:
def load_and_convert(vid):
    """Load LiTS volume and convert to uint8 channel-last for fastloops."""
    ct = np.load(os.path.join(DATA_ROOT, "ct", f"volume-{vid}.npy")).astype(np.float32)
    seg = np.load(os.path.join(DATA_ROOT, "seg", f"segmentation-{vid}.npy")).astype(np.int32)
    ct_u8 = np.clip(ct, HU_MIN, HU_MAX)
    ct_u8 = ((ct_u8 - HU_MIN) / (HU_MAX - HU_MIN) * 255).round().astype(np.uint8)
    ct_u8 = np.ascontiguousarray(ct_u8[..., np.newaxis])  # channel-last
    return ct, seg, ct_u8


def discover_volumes():
    """Find LiTS volumes that have tumor."""
    ct_dir = os.path.join(DATA_ROOT, "ct")
    vids = []
    for f in sorted(os.listdir(ct_dir)):
        m = re.match(r"volume-(\d+)\.npy", f)
        if m:
            vid = int(m.group(1))
            seg_path = os.path.join(DATA_ROOT, "seg", f"segmentation-{vid}.npy")
            if os.path.exists(seg_path):
                seg = np.load(seg_path)
                if (seg == 2).sum() > 0:
                    vids.append(vid)
    return sorted(vids)


all_vids = discover_volumes()
pr(f"Found {len(all_vids)} LiTS volumes with tumor")

# Use first 10 for diagnostics
diag_vids = all_vids[:10]
pr(f"Diagnostic volumes: {diag_vids}")

Found 118 LiTS volumes with tumor


Diagnostic volumes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


## Step 2: Liver ROI crop + protected region helpers

In [3]:
def bbox_from_mask(mask, margin=32):
    """Bounding box with margin from binary mask."""
    coords = np.argwhere(mask)
    z0, y0, x0 = coords.min(axis=0)
    z1, y1, x1 = coords.max(axis=0) + 1
    z0 = max(z0 - margin, 0)
    y0 = max(y0 - margin, 0)
    x0 = max(x0 - margin, 0)
    z1 = min(z1 + margin, mask.shape[0])
    y1 = min(y1 + margin, mask.shape[1])
    x1 = min(x1 + margin, mask.shape[2])
    return (slice(z0, z1), slice(y0, y1), slice(x0, x1))


def make_intensity_protected(ct_u8_crop, organ_crop):
    """Create intensity-based protected candidate region inside organ.
    
    Hypodense voxels relative to liver parenchyma are suspicious for tumor.
    Does NOT use seg==2 (tumor GT) — only organ mask + intensity.
    """
    liver_vals = ct_u8_crop[..., 0][organ_crop]
    liver_mean = liver_vals.mean()
    liver_std = liver_vals.std()
    
    # Hypodense candidate: below 0.5 std of liver mean, inside organ
    candidate = organ_crop & (ct_u8_crop[..., 0] < liver_mean - 0.5 * liver_std)
    
    # Dilate to cover tumor boundary
    protected = binary_dilation(candidate, iterations=2)
    return protected.astype(np.uint8)


def make_gt_protected(seg_crop):
    """GT tumor protection (upper bound diagnostic only)."""
    gt_tumor = seg_crop == 2
    protected = binary_dilation(gt_tumor, iterations=3)
    return protected.astype(np.uint8)


def oracle_dice_multi(labels_np, seg):
    """Compute oracle Dice at multiple overlap thresholds."""
    flat = labels_np.ravel()
    gt = (seg.ravel() == 2).astype(np.float64)
    gt_total = int(gt.sum())
    valid = flat >= 0
    result = {}
    if gt_total == 0 or not valid.any():
        for th in [0.01, 0.05, 0.10, 0.25, 0.50]:
            result[f"oracle_{th:.2f}"] = 0.0
        result["tumor_deleted_pct"] = 0.0
        return result
    max_id = int(flat[valid].max())
    tc = np.bincount(flat[valid], weights=gt[valid], minlength=max_id + 1)
    total_c = np.bincount(flat[valid], minlength=max_id + 1)
    overlap = tc / np.maximum(total_c, 1)
    gt_mask = seg == 2
    for th in [0.01, 0.05, 0.10, 0.25, 0.50]:
        tumor_sids = np.where(overlap > th)[0]
        lut = np.zeros(max_id + 1, dtype=np.int32)
        lut[tumor_sids] = 1
        pred = np.where(valid, lut[flat], 0).reshape(labels_np.shape).astype(bool)
        inter = int((pred & gt_mask).sum())
        result[f"oracle_{th:.2f}"] = 2.0 * inter / (pred.sum() + gt_mask.sum() + 1e-8)
    del_tumor = int(gt[~valid].sum())
    result["tumor_deleted_pct"] = del_tumor / max(gt_total, 1) * 100.0
    return result


pr("Helpers defined.")

Helpers defined.


## Step 3: Run 4-mode oracle diagnostics on 10 volumes

For each volume, run coarsening under 4 modes and compare oracle Dice.

In [4]:
def run_coarsen(ct_u8, protect_mask, psi, alpha, use_default_deletion=False):
    """Run fastloops coarsening with optional protection mask.
    
    use_default_deletion=True: let fastloops use its built-in size thresholds
    use_default_deletion=False: disable deletion entirely (delete_small=0, delete_large=n+1)
    
    When protect_mask is provided, protected supernodes survive deletion
    regardless of the deletion setting.
    """
    n_vox = ct_u8[..., 0].size
    
    if use_default_deletion:
        # Let fastloops pick defaults: small=log2(n), large=n^0.8
        kwargs = dict(
            merge_distance=psi, cut_distance=alpha,
            # Omit delete params -> fastloops uses defaults
            connectivity="faces",
        )
    else:
        kwargs = dict(
            merge_distance=psi, cut_distance=alpha,
            delete_small_node_max_size=0,
            delete_large_node_min_size=n_vox + 1,
            delete_value_min=0, delete_value_max=255,
            connectivity="faces",
        )
    
    if protect_mask is not None:
        pm = np.ascontiguousarray(protect_mask)
        nf, ei, ef, labels, adj = fastloops.merge_and_cut_protected(ct_u8, pm, **kwargs)
    else:
        nf, ei, ef, labels, adj = fastloops.merge_and_cut(ct_u8, **kwargs)
    return np.asarray(labels), nf, ei, ef


PSI, ALPHA = 5, 25
pr(f"psi={PSI}, alpha={ALPHA}")
pr(f"Modes A/B: DEFAULT deletion (aggressive)")
pr(f"Modes C/D: DEFAULT deletion + protection saves tumor supernodes")

psi=5, alpha=25


Modes A/B: DEFAULT deletion (aggressive)


Modes C/D: DEFAULT deletion + protection saves tumor supernodes


In [5]:
all_results = []

for vid in diag_vids:
    ct_raw, seg, ct_u8_full = load_and_convert(vid)
    n_tumor = int((seg == 2).sum())
    n_liver = int((seg == 1).sum())
    pr(f"\n--- vol-{vid}: shape={ct_raw.shape} liver={n_liver:,} tumor={n_tumor:,} ---")
    
    # Build organ ROI (liver + tumor)
    organ_mask = (seg == 1) | (seg == 2)
    slc = bbox_from_mask(organ_mask, margin=32)
    ct_crop = ct_raw[slc]
    seg_crop = seg[slc]
    organ_crop = organ_mask[slc]
    
    # Preprocess crop to uint8 channel-last
    ct_u8_crop = np.clip(ct_crop, HU_MIN, HU_MAX)
    ct_u8_crop = ((ct_u8_crop - HU_MIN) / (HU_MAX - HU_MIN) * 255).round().astype(np.uint8)
    ct_u8_crop = np.ascontiguousarray(ct_u8_crop[..., np.newaxis])
    
    pr(f"  Crop shape: {ct_crop.shape} ({ct_crop.size:,} voxels, "
       f"{ct_crop.size/ct_raw.size*100:.1f}% of full)")
    
    # Build protection masks
    protect_intensity = make_intensity_protected(ct_u8_crop, organ_crop)
    protect_gt = make_gt_protected(seg_crop)
    
    # Debug: check protection quality (does NOT leak into graph)
    gt_tumor_crop = seg_crop == 2
    if gt_tumor_crop.sum() > 0:
        int_recall = (protect_intensity.astype(bool) & gt_tumor_crop).sum() / gt_tumor_crop.sum()
        int_precision = (protect_intensity.astype(bool) & gt_tumor_crop).sum() / max(protect_intensity.sum(), 1)
        pr(f"  Intensity protection: recall={int_recall:.3f} precision={int_precision:.3f}")
    
    row = {"vid": vid, "n_tumor": n_tumor, "crop_shape": list(ct_crop.shape)}
    
    # All modes use DEFAULT deletion.
    # A/B have no protect mask -> deletion kills tumor supernodes freely
    # C/D have protect mask -> protected supernodes survive deletion
    for mode, label, ct_input, pm in [
        ("A", "full_default_del", ct_u8_full, None),
        ("B", "crop_default_del", ct_u8_crop, None),
        ("C", "crop_int_protect", ct_u8_crop, protect_intensity),
        ("D", "crop_gt_protect", ct_u8_crop, protect_gt),
    ]:
        t0 = time.time()
        labels, nf, ei, ef = run_coarsen(ct_input, pm, PSI, ALPHA, use_default_deletion=True)
        dt = time.time() - t0
        
        seg_eval = seg if mode == "A" else seg_crop
        od = oracle_dice_multi(labels, seg_eval)
        n_sn = nf.shape[0]
        n_edges = ei.shape[1]
        
        row[f"{mode}_sn"] = n_sn
        row[f"{mode}_edges"] = n_edges
        row[f"{mode}_time"] = round(dt, 2)
        for k, v in od.items():
            row[f"{mode}_{k}"] = round(v, 4)
        
        pr(f"  Mode {mode} ({label:20s}): {n_sn:>7,} SN  {n_edges:>6,} edges  "
           f"o@.10={od['oracle_0.10']:.4f}  o@.25={od['oracle_0.25']:.4f}  "
           f"o@.50={od['oracle_0.50']:.4f}  del={od['tumor_deleted_pct']:.1f}%  {dt:.1f}s")
    
    all_results.append(row)

pr("\nDone.")


--- vol-0: shape=(28, 256, 256) liver=138,634 tumor=704 ---


  Crop shape: (28, 178, 219) (1,091,496 voxels, 59.5% of full)


  Intensity protection: recall=1.000 precision=0.005


  Mode A (full_default_del    ):     764 SN   1,219 edges  o@.10=0.1296  o@.25=0.1296  o@.50=0.1296  del=78.7%  0.2s


  Mode B (crop_default_del    ):     768 SN   1,291 edges  o@.10=0.0709  o@.25=0.0709  o@.50=0.0709  del=81.1%  0.1s


...

  Mode A (full_default_del    ):   5,845 SN  11,054 edges  o@.10=0.3230  o@.25=0.3232  o@.50=0.3172  del=78.5%  1.1s


  Mode B (crop_default_del    ):   5,709 SN  10,909 edges  o@.10=0.3349  o@.25=0.3347  o@.50=0.3186  del=77.8%  0.6s


  Mode C (crop_int_protect    ): 223,568 SN  846,597 edges  o@.10=0.8861  o@.25=0.9127  o@.50=0.9208  del=0.0%  0.9s


  Mode D (crop_gt_protect     ):  12,003 SN  34,290 edges  o@.10=0.8843  o@.25=0.9158  o@.50=0.9228  del=0.0%  0.6s



Done.


## Step 4: Summary table

In [6]:
# Summary across all volumes
pr(f"\n{'='*100}")
pr(f"  SUMMARY: psi={PSI} alpha={ALPHA}, {len(all_results)} volumes")
pr(f"{'='*100}")
pr(f"{'Mode':<6s} {'Description':<28s} {'o@.10':>7s} {'o@.25':>7s} {'o@.50':>7s} "
   f"{'del%':>6s} {'SN':>10s} {'edges':>8s} {'time':>6s}")
pr("-" * 100)

for mode, desc in [("A", "Full CT, no protect"),
                    ("B", "Liver crop, no protect"),
                    ("C", "Liver crop + intensity protect"),
                    ("D", "Liver crop + GT protect (UB)")]:
    o10 = np.mean([r[f"{mode}_oracle_0.10"] for r in all_results])
    o25 = np.mean([r[f"{mode}_oracle_0.25"] for r in all_results])
    o50 = np.mean([r[f"{mode}_oracle_0.50"] for r in all_results])
    dl = np.mean([r[f"{mode}_tumor_deleted_pct"] for r in all_results])
    sn = np.mean([r[f"{mode}_sn"] for r in all_results])
    ed = np.mean([r[f"{mode}_edges"] for r in all_results])
    tm = np.mean([r[f"{mode}_time"] for r in all_results])
    pr(f"  {mode:<4s} {desc:<28s} {o10:7.4f} {o25:7.4f} {o50:7.4f} "
       f"{dl:5.1f}% {sn:10,.0f} {ed:8,.0f} {tm:5.1f}s")

pr(f"\nInterpretation:")
pr(f"  B > A  =>  full-volume coarsening was hurting")
pr(f"  C > B  =>  intensity protection helps")
pr(f"  D >> C =>  candidate proposal is missing tumor")
pr(f"  D bad  =>  graph/lifting/model bug remains")

# Save results
with open(os.path.join(RESULTS_DIR, "oracle_diagnostics.json"), "w") as f:
    json.dump({"psi": PSI, "alpha": ALPHA, "results": all_results}, f, indent=2)
pr(f"\nSaved to {RESULTS_DIR}/oracle_diagnostics.json")

  SUMMARY: psi=5 alpha=25, 10 volumes


Mode   Description                    o@.10   o@.25   o@.50   del%         SN    edges   time


----------------------------------------------------------------------------------------------------


  A    Full CT, no protect           0.2829  0.2823  0.2732  77.4%      5,159    9,306   0.9s


  B    Liver crop, no protect        0.2809  0.2808  0.2709  77.2%      4,908    8,956   0.5s


...

  B > A  =>  full-volume coarsening was hurting


  C > B  =>  intensity protection helps


  D >> C =>  candidate proposal is missing tumor


  D bad  =>  graph/lifting/model bug remains



Saved to /home/ud3d4/Desktop/SWOG/results/semir_lits_twostage/oracle_diagnostics.json


## Step 5: Build PyG graphs (Mode C) + Train GINE + Evaluate voxel Dice

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GINEConv, BatchNorm

# ---- Feature extraction (from semir_lits_v7.py / Luke's crate) ----

def _layout(C):
    return dict(area=0, s=[1, 2, 3],
                cov=[(4, 0, 0), (5, 1, 1), (6, 2, 2), (7, 0, 1), (8, 0, 2), (9, 1, 2)],
                chan0=10, boundary=10 + C + 6, D=3)

def node_invariants(node_feats, C=1, eps=1e-6):
    f = node_feats.astype(np.float64)
    L = _layout(C); D = L["D"]; N = f.shape[0]
    V = f[:, L["area"]]; Vsafe = np.maximum(V, 1.0)
    mean_coord = np.stack([f[:, c] for c in L["s"]], axis=1) / Vsafe[:, None]
    cov = np.zeros((N, D, D))
    for col, i, j in L["cov"]:
        cij = f[:, col] / Vsafe - mean_coord[:, i] * mean_coord[:, j]
        cov[:, i, j] = cij; cov[:, j, i] = cij
    w = np.linalg.eigvalsh(cov); w = np.clip(w, 0.0, None)
    _, vec = np.linalg.eigh(cov); principal = vec[..., -1]
    trace = w.sum(axis=1); degenerate = trace < eps
    denom = w[:, 2] + eps
    shape = np.stack([(w[:, 2] - w[:, 1]) / denom,
                      (w[:, 1] - w[:, 0]) / denom,
                      w[:, 0] / denom], axis=1)
    shape[degenerate] = 0.0
    line_like = np.where(degenerate, 0.0, shape[:, 0])
    chan = f[:, L["chan0"]:L["chan0"] + C] / Vsafe[:, None] / 255.0
    compactness = f[:, L["boundary"]] / np.power(Vsafe, (D - 1.0) / D)
    elongation = np.where(w[:, 0] > eps, w[:, 2] / (w[:, 0] + eps), 1.0)
    elongation = np.clip(elongation, 1.0, 100.0)
    return dict(V=V, surface=f[:, L["boundary"]], centroid=mean_coord,
                eig=w, shape=shape, line_like=line_like, principal=principal,
                chan=chan, compactness=compactness, elongation=elongation)

def edge_invariants(node_feats, edge_index, edge_feats, C=1, eps=1e-6):
    inv = node_invariants(node_feats, C, eps)
    a = edge_index[0].astype(np.int64); b = edge_index[1].astype(np.int64)
    ef = edge_feats.astype(np.float64); blsafe = np.maximum(ef[:, 0], 1.0)
    size_contrast = np.abs(inv["V"][a] - inv["V"][b]) / (inv["V"][a] + inv["V"][b] + eps)
    bfrac_a = ef[:, 0] / (inv["surface"][a] + eps)
    bfrac_b = ef[:, 0] / (inv["surface"][b] + eps)
    mean_contrast = np.abs(inv["chan"][a] - inv["chan"][b])
    shape_dissim = np.abs(inv["shape"][a] - inv["shape"][b])
    axis_align = (np.abs(np.sum(inv["principal"][a] * inv["principal"][b], axis=1))
                  * np.minimum(inv["line_like"][a], inv["line_like"][b]))
    bcontrast = (ef[:, 1] / blsafe) / 255.0
    cut_frac = ef[:, 3] / blsafe
    cols = [size_contrast[:, None], bfrac_a[:, None], bfrac_b[:, None],
            mean_contrast if mean_contrast.ndim > 1 else mean_contrast[:, None],
            shape_dissim, axis_align[:, None], bcontrast[:, None], cut_frac[:, None]]
    return np.concatenate(cols, axis=1).astype(np.float32)

def compute_intensity_std(labels_np, ct_u8):
    flat = labels_np.ravel(); valid = flat >= 0
    if not valid.any(): return np.array([], dtype=np.float32)
    max_id = int(flat[valid].max())
    vals = ct_u8[..., 0].ravel().astype(np.float64) / 255.0
    counts = np.bincount(flat[valid], minlength=max_id + 1).astype(np.float64)
    sums = np.bincount(flat[valid], weights=vals[valid], minlength=max_id + 1)
    sq_sums = np.bincount(flat[valid], weights=vals[valid] ** 2, minlength=max_id + 1)
    mean = sums / np.maximum(counts, 1.0)
    var = sq_sums / np.maximum(counts, 1.0) - mean ** 2
    return np.sqrt(np.maximum(var, 0.0)).astype(np.float32)

def build_pyg_graph(raw_nf, raw_ei, raw_ef, labels_np, seg, ct_u8, overlap_th, C=1):
    n_sn = raw_nf.shape[0]
    inv = node_invariants(raw_nf, C)
    int_std = compute_intensity_std(labels_np, ct_u8)
    if len(int_std) < n_sn: int_std = np.pad(int_std, (0, n_sn - len(int_std)))
    int_std = int_std[:n_sn]
    principal = inv["principal"].astype(np.float32)
    if len(principal):
        max_comp = np.argmax(np.abs(principal), axis=1)
        signs = np.sign(principal[np.arange(len(principal)), max_comp])
        signs[signs == 0] = 1
        principal = principal * signs[:, None]
    x = np.column_stack([
        np.log1p(inv["V"]), np.log1p(inv["surface"]),
        inv["compactness"], inv["elongation"],
        principal[:, 0], principal[:, 1], principal[:, 2],
        inv["chan"][:, 0], int_std,
    ]).astype(np.float32)
    for col in range(x.shape[1]):
        mu, sigma = float(x[:, col].mean()), float(x[:, col].std())
        if sigma > 1e-8: x[:, col] = (x[:, col] - mu) / sigma
        else: x[:, col] = 0.0
    if raw_ei.shape[1] > 0:
        ea = edge_invariants(raw_nf, raw_ei, raw_ef, C)
        ei_fwd = torch.tensor(raw_ei, dtype=torch.long)
        ei_rev = torch.stack([ei_fwd[1], ei_fwd[0]])
        edge_index = torch.cat([ei_fwd, ei_rev], dim=1)
        edge_attr = torch.tensor(np.concatenate([ea, ea]), dtype=torch.float32)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 10), dtype=torch.float32)
    flat = labels_np.ravel(); valid = flat >= 0
    gt = (seg.ravel() == 2).astype(np.float64)
    max_id = int(flat[valid].max()) if valid.any() else -1
    y = np.zeros(n_sn, dtype=np.int64)
    if max_id >= 0:
        tc = np.bincount(flat[valid], weights=gt[valid], minlength=max_id + 1)
        total_c = np.bincount(flat[valid], minlength=max_id + 1)
        overlap = tc / np.maximum(total_c, 1)
        y[:min(n_sn, len(overlap))] = (overlap[:n_sn] >= overlap_th).astype(np.int64)
    return Data(x=torch.tensor(x, dtype=torch.float32),
                edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor(y, dtype=torch.long))

# ---- GINE model ----

class GINE(nn.Module):
    def __init__(self, nd, ed, h=128):
        super().__init__()
        self.ep = nn.Linear(ed, h)
        def mlp(d): return nn.Sequential(nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Linear(h, h))
        self.c1 = GINEConv(mlp(nd), edge_dim=h); self.b1 = BatchNorm(h)
        self.c2 = GINEConv(mlp(h), edge_dim=h); self.b2 = BatchNorm(h)
        self.c3 = GINEConv(mlp(h), edge_dim=h); self.b3 = BatchNorm(h)
        self.head = nn.Linear(h, 2)
    def forward(self, x, ei, ea):
        if ea is not None and ea.numel() > 0: ea = self.ep(ea)
        else:
            n = x.size(0); ei = torch.stack([torch.arange(n, device=x.device)]*2)
            ea = torch.zeros(n, self.ep.out_features, device=x.device)
        x = F.relu(self.b1(self.c1(x, ei, ea)))
        x = F.relu(self.b2(self.c2(x, ei, ea)))
        x = F.relu(self.b3(self.c3(x, ei, ea)))
        return self.head(x)

pr("Feature extraction + GINE model defined.")

/home/ud3d4/.conda/envs/llmft/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Feature extraction + GINE model defined.


In [8]:
# ---- Build Mode C graphs for all 118 volumes ----

perm = np.random.permutation(len(all_vids))
n_train = int(0.7 * len(all_vids)); n_val = int(0.15 * len(all_vids))
train_ids = sorted([all_vids[i] for i in perm[:n_train]])
val_ids = sorted([all_vids[i] for i in perm[n_train:n_train + n_val]])
test_ids = sorted([all_vids[i] for i in perm[n_train + n_val:]])
pr(f"Split: {len(train_ids)} train / {len(val_ids)} val / {len(test_ids)} test")

OVERLAP_TH = 0.10
MAX_NODES_GPU = 500_000  # skip volumes too large for GPU
GRAPH_CACHE = "/dev/shm/semir_twostage_graphs"
os.makedirs(GRAPH_CACHE, exist_ok=True)

graphs = {}; label_maps = {}; seg_maps = {}; slc_maps = {}

for vid in train_ids + val_ids + test_ids:
    cache_g = os.path.join(GRAPH_CACHE, f"graph_{vid}.pt")
    cache_l = os.path.join(GRAPH_CACHE, f"labels_{vid}.npy")
    cache_s = os.path.join(GRAPH_CACHE, f"seg_{vid}.npy")
    cache_slc = os.path.join(GRAPH_CACHE, f"slc_{vid}.json")
    
    if os.path.exists(cache_g):
        g = torch.load(cache_g, weights_only=False)
        lnp = np.load(cache_l); s = np.load(cache_s)
        with open(cache_slc) as f: slc_info = json.load(f)
        slc = tuple(slice(s[0], s[1]) for s in slc_info)
    else:
        ct_raw, seg, ct_u8_full = load_and_convert(vid)
        
        # Liver crop
        organ_mask = (seg == 1) | (seg == 2)
        slc = bbox_from_mask(organ_mask, margin=32)
        ct_crop = ct_raw[slc]
        seg_crop = seg[slc]
        organ_crop = organ_mask[slc]
        
        ct_u8_crop = np.clip(ct_crop, HU_MIN, HU_MAX)
        ct_u8_crop = ((ct_u8_crop - HU_MIN) / (HU_MAX - HU_MIN) * 255).round().astype(np.uint8)
        ct_u8_crop = np.ascontiguousarray(ct_u8_crop[..., np.newaxis])
        
        # Mode C: intensity-protected coarsening with default deletion
        protect = make_intensity_protected(ct_u8_crop, organ_crop)
        lnp, raw_nf, raw_ei, raw_ef = run_coarsen(
            ct_u8_crop, protect, PSI, ALPHA, use_default_deletion=True)
        s = seg_crop
        
        g = build_pyg_graph(raw_nf, raw_ei, raw_ef, lnp, s, ct_u8_crop, OVERLAP_TH)
        
        torch.save(g, cache_g); np.save(cache_l, lnp); np.save(cache_s, s)
        slc_info = [[int(sl.start), int(sl.stop)] for sl in slc]
        with open(cache_slc, "w") as f: json.dump(slc_info, f)
    
    graphs[vid] = g; label_maps[vid] = lnp; seg_maps[vid] = s; slc_maps[vid] = slc
    split = "train" if vid in train_ids else ("val" if vid in val_ids else "test")
    n_tu = int((g.y == 1).sum()); n_bg = int((g.y == 0).sum())
    pr(f"  vol-{vid:>3d} [{split:>5s}]: {g.num_nodes:>7,} nodes ({n_tu:>5,} tu) "
       f"{g.num_edges:>8,} edges")

pr(f"\nTotal graphs: {len(graphs)}")

Split: 82 train / 17 val / 19 test


  vol-  0 [train]:  41,785 nodes (  258 tu)  266,134 edges


  vol-  3 [train]: 342,637 nodes (  273 tu) 2,658,674 edges


  vol-  4 [train]: 360,855 nodes (132,533 tu) 2,558,010 edges


  vol-  5 [train]: 242,988 nodes (   62 tu) 1,820,488 edges


...

  vol-112 [ test]: 318,050 nodes (  191 tu) 2,326,992 edges


  vol-118 [ test]: 275,778 nodes (43,469 tu) 2,076,406 edges


  vol-126 [ test]: 309,372 nodes (  634 tu) 2,383,916 edges


  vol-130 [ test]: 573,793 nodes (149,927 tu) 4,170,812 edges



Total graphs: 118


In [9]:
# ---- Train GINE ----

device = "cuda:0" if torch.cuda.is_available() else "cpu"
pr(f"Device: {device}")
if torch.cuda.is_available():
    pr(f"GPU: {torch.cuda.get_device_name(0)}")

trainable = [v for v in train_ids if v in graphs and graphs[v].num_nodes <= MAX_NODES_GPU]
val_usable = [v for v in val_ids if v in graphs]
pr(f"Trainable: {len(trainable)}/{len(train_ids)}, Val: {len(val_usable)}/{len(val_ids)}")

# Class weights
total_pos = sum(int((graphs[v].y == 1).sum()) for v in trainable)
total_neg = sum(int((graphs[v].y == 0).sum()) for v in trainable)
ratio = total_neg / max(total_pos, 1)
eff = min(np.sqrt(ratio), 30.0)
class_weight = torch.tensor([1.0, eff], dtype=torch.float32).to(device)
pr(f"Class weight: [1.0, {eff:.1f}] (ratio: {ratio:.0f}:1)")

nd = graphs[trainable[0]].x.shape[1]
ed = graphs[trainable[0]].edge_attr.shape[1] if graphs[trainable[0]].edge_attr.numel() > 0 else 10

model = GINE(nd, ed).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

PATIENCE = 15; EPOCHS = 200; VAL_EVERY = 3
best_dice, best_state, wait = -1.0, None, 0
history = {"train_loss": [], "val_dice": []}

for epoch in range(1, EPOCHS + 1):
    model.train(); epoch_loss = 0.0; processed = 0
    for vid in np.random.permutation(trainable):
        try:
            g = graphs[vid].to(device)
            opt.zero_grad()
            logits = model(g.x, g.edge_index, g.edge_attr)
            loss = F.cross_entropy(logits, g.y, weight=class_weight)
            loss.backward(); opt.step()
            epoch_loss += loss.item(); processed += 1
            del g, logits, loss
        except torch.cuda.OutOfMemoryError:
            try: del g
            except: pass
            torch.cuda.empty_cache(); continue
        torch.cuda.empty_cache()
    
    if processed == 0:
        pr("ERROR: No graphs processed (all OOM)"); break
    mean_loss = epoch_loss / processed
    history["train_loss"].append(mean_loss)
    
    if epoch % VAL_EVERY == 0 or epoch <= 3:
        model.eval(); tp = fp = fn = 0
        with torch.no_grad():
            for vid in val_usable:
                g = graphs[vid]; lnp = label_maps[vid]; s = seg_maps[vid]
                try:
                    gd = g.to(device)
                    preds = model(gd.x, gd.edge_index, gd.edge_attr).argmax(dim=1).cpu().numpy()
                    del gd; torch.cuda.empty_cache()
                except (torch.cuda.OutOfMemoryError, RuntimeError):
                    try: del gd
                    except: pass
                    torch.cuda.empty_cache()
                    mc = model.cpu()
                    preds = mc(g.x, g.edge_index, g.edge_attr).argmax(dim=1).numpy()
                    model.to(device)
                # Lift to voxel level
                flat = lnp.ravel(); valid = flat >= 0
                if not valid.any(): continue
                mid = int(flat[valid].max())
                lut = np.zeros(mid + 1, dtype=np.int8)
                lut[:min(len(preds), mid + 1)] = preds[:min(len(preds), mid + 1)]
                pm = np.where(valid, lut[flat], 0).reshape(lnp.shape).astype(bool)
                gm = s == 2
                inter = int((pm & gm).sum())
                tp += inter; fp += int(pm.sum()) - inter; fn += int(gm.sum()) - inter
        
        vd = 2 * tp / (2 * tp + fp + fn + 1e-8)
        history["val_dice"].append(vd)
        if vd > best_dice:
            best_dice = vd
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0; marker = " *"
        else:
            wait += 1; marker = ""
        pr(f"  Epoch {epoch:3d}  loss={mean_loss:.4f}  val_dice={vd:.4f}  ({processed} vols){marker}")
        if wait >= PATIENCE:
            pr(f"  Early stop epoch {epoch}, best val Dice={best_dice:.4f}"); break
    else:
        pr(f"  Epoch {epoch:3d}  loss={mean_loss:.4f}  ({processed} vols)")

if best_state: model.load_state_dict(best_state)
pr(f"\nBest val Dice: {best_dice:.4f}")
torch.save(best_state or model.state_dict(), os.path.join(RESULTS_DIR, "model.pt"))

Device: cuda:0


GPU: NVIDIA L40S


Trainable: 72/82, Val: 17/17


Class weight: [1.0, 5.3] (ratio: 29:1)


  Epoch   1  loss=0.4088  val_dice=0.0000  (72 vols) *


...

  Epoch  61  loss=0.1823  (72 vols)


  Epoch  62  loss=0.2088  (72 vols)


  Epoch  63  loss=0.1594  val_dice=0.0937  (72 vols)


  Early stop epoch 63, best val Dice=0.3209



Best val Dice: 0.3209


## Step 6: Final evaluation — voxel-level Dice per split

In [10]:
# ---- Final evaluation: voxel-level Dice on all splits ----

model.eval()
results = []

for vid in sorted(graphs.keys()):
    g = graphs[vid]; lnp = label_maps[vid]; s = seg_maps[vid]
    with torch.no_grad():
        try:
            gd = g.to(device)
            preds = model(gd.x, gd.edge_index, gd.edge_attr).argmax(dim=1).cpu().numpy()
            del gd; torch.cuda.empty_cache()
        except:
            torch.cuda.empty_cache()
            preds = model.cpu()(g.x, g.edge_index, g.edge_attr).argmax(dim=1).numpy()
            model.to(device)
    
    flat = lnp.ravel(); valid = flat >= 0
    pm = np.zeros(lnp.shape, dtype=bool)
    if valid.any():
        mid = int(flat[valid].max())
        lut = np.zeros(mid + 1, dtype=np.int8)
        lut[:min(len(preds), mid + 1)] = preds[:min(len(preds), mid + 1)]
        pm = np.where(valid, lut[flat], 0).reshape(lnp.shape).astype(bool)
    
    gm = s == 2
    inter = int((gm & pm).sum())
    dice = 2.0 * inter / (gm.sum() + pm.sum() + 1e-8)
    rec = inter / (gm.sum() + 1e-8)
    prec = inter / (pm.sum() + 1e-8) if pm.sum() > 0 else 0.0
    
    split = "train" if vid in train_ids else ("val" if vid in val_ids else "test")
    results.append({"vid": vid, "split": split, "dice": float(dice),
                     "recall": float(rec), "precision": float(prec),
                     "n_nodes": g.num_nodes, "n_tumor_nodes": int((g.y == 1).sum()),
                     "gt_voxels": int(gm.sum()), "pred_voxels": int(pm.sum())})

# Summary
pr(f"\n{'='*80}")
pr(f"  VOXEL-LEVEL DICE RESULTS (Mode C: liver crop + intensity protection)")
pr(f"{'='*80}")
pr(f"  {'Split':>5s}  {'Dice':>8s}  {'Recall':>8s}  {'Prec':>8s}  {'N':>4s}")
pr(f"  {'-'*40}")
for split in ["train", "val", "test"]:
    scores = [r for r in results if r["split"] == split]
    if scores:
        d = np.mean([r["dice"] for r in scores])
        r = np.mean([r["recall"] for r in scores])
        p = np.mean([r["precision"] for r in scores])
        pr(f"  {split:>5s}  {d:8.4f}  {r:8.4f}  {p:8.4f}  {len(scores):4d}")

pr(f"\n  Paper target: LiTS tumor Dice = 0.891 +/- 0.007")
pr(f"  Best val Dice during training: {best_dice:.4f}")

# Save
with open(os.path.join(RESULTS_DIR, "final_results.json"), "w") as f:
    json.dump({"params": {"psi": PSI, "alpha": ALPHA, "hu": [HU_MIN, HU_MAX],
                           "overlap_th": OVERLAP_TH, "mode": "C_intensity_protect"},
               "best_val_dice": float(best_dice),
               "epochs": len(history["train_loss"]),
               "history": history, "results": results}, f, indent=2)
pr(f"\n  Saved to {RESULTS_DIR}/final_results.json")

  VOXEL-LEVEL DICE RESULTS (Mode C: liver crop + intensity protection)


  Split      Dice    Recall      Prec     N


  ----------------------------------------


  train    0.1761    0.2352    0.2397    82


    val    0.1965    0.2163    0.3427    17


   test    0.2327    0.3000    0.3428    19



  Paper target: LiTS tumor Dice = 0.891 +/- 0.007


  Best val Dice during training: 0.3209



  Saved to /home/ud3d4/Desktop/SWOG/results/semir_lits_twostage/final_results.json
